# Práctica 3: Programación en Apache Spark

En esta práctica vamos a usar Spark sobre el dataset que empleamos en la práctica anterior (https://www.kaggle.com/datasets/mkechinov/ecommerce-behavior-data-from-multi-category-store). Recordad que se trata de un log sobre el comportamiento de los usuarios cuando visitan la web de una tienda online.  
En esta práctica vamos a realizar ejemplos similares a los realizados en la práctica anterior, pero ahora usando Spark. Concretamente usaremos PySpark. Para ello, primero llevaremos a cabo la instalación de PySpark, probaremos ejemplos en local y, posteriormente, usaremos el cluster de Hadoop para probar los ejemplos que hemos creado. 

---
## 1. Instalación y prueba de PySpark en local

De forma similar a como hemos hecho en la práctica de MrJob, vamos a llevar a cabo la instalación de PySpark para poder hacer pruebas en local con el fichero `2019-Oct_10k.txt`.  
Para la instalación, hemos creado un nuevo entorno en environment/p3 y hemos añadido ficheros en la carpeta practicas/p3 del repo ppd-public. Se explica a continuación cómo hacer pull del repo e instalar todo lo necesario.

1. Tenemos que hacer pull del repo para traernos los nuevos ficheros que hemos añadido en esta pr:
```bash
git pull origin main
```
2. Desde la carpeta environment/p3 creamos el nuevo .env
```bash
uv lock
uv sync --locked
```
3. Tenemos que comprobar si tenemos java instalado, dado que, como hemos visto en las clases de teoría, Spark necesita Java. En el caso de que no esté instalado, debemos instalarlo. 
```bash
    java --version
```
Si la salida no devuelve la versión de java instalada, tenemos que ejecutar:
```bash
    sudo apt install openjdk-17-jdk
```
4. Activamos el entorno:
```bash
    source .venv/bin/activate
```
5. Ahora podemos ejecutar el ejemplo con PySpark que tenemos en practicas/p3. El primero que vamos a probar es **Contador de eventos**.  
Para no duplicar ficheros en el repo recuerda que debes copiar las entradas de la práctica de MapReduce a la de Spark.  
Es decir, tenemos el fichero ej1_event_count_pyspark.py en practicas/p2/2_3_spark junto con el fichero `2019-Oct_10k.txt` de la p2/2_2_mapreduce.  
Podemos entonces ejecutar:  

```bash
    spark-submit ej1_event_count_pyspark.py
```
Podemos entonces observar la salida de la ejecución en Spark. Si queremos eliminar la cantidad de logs que muestra, podemos derivar la salida a /dev/null con:
```bash
    spark-submit ej1_event_count_pyspark.py 2>/dev/null
```
6. Podemos ver la interfaz gráfica de Spark abriendo en un navegador web http://localhost:4040/  
Sin embargo, veremos que el servicio web se cierra y la web no está disponible cuando termina PySpark. Para ello podemos añadir la siguiente línea al ejemplo que nos deja la ejecución abierta hasta que pulsemos Intro.   
`input("Pulsa ENTER para cerrar Spark...")`


## 2. Uso de PySpark en el cluster de Hadoop

Ahora vamos a probar el mismo funcionamiento pero dentro del cluster Hadoop que nos creamos en la práctica 2.  
Es importante comentar la linea de `input("Pulsa ENTER para cerrar Spark...")`

1. Copiamos los dos ficheros al docker
```bash
docker cp ej1_event_count_pyspark.py workbench:/home/luser
docker cp 2019-Oct_10k.txt workbench:/home/luser
```

2. Nos conectamos al workbench
```bash
docker compose exec workbench bash
```

3. Cambiamos al usuario luser
```bash
chown luser:hadoop /home/luser/ej1_event_count_pyspark.py  
su - luser
```

4. Cargamos el fichero a hdfs
```bash
hdfs dfs -put -f 2019-Oct_10k.txt /user/luser
```

5. Podemos ejecutar con el comando 
```bash
spark-submit --master yarn ej1_event_count_pyspark.py hdfs://namenode:9000/user/luser/2019-Oct_10k.txt
```

Podeis ver más detalles sobre como ejecutar en el cluster Spark en el notebook de teoría 3-08-Spark-en-un-cluster.ipynb

## 3. Ejercicios

Se propone la realización de 4 scripts. Los scripts pueden realizarse usando Notebooks o bien como scripts de Python individuales que se puedan enviar con `spark-submit`. Para ello vamos a usar los dos ficheros de la práctica anterior, `2019-Oct_10k.txt` y `2019-Oct_600k.txt`
### Normas:
- Los scripts deben incluir comentarios que expliquen los pasos realizados.
- La salida de los scripts debe seguir el formato indicado en cada uno de los ejercicios (incluyendo el nombre y orden de las columnas).
- Se debe entregar un fichero comprimido con los scripts de la práctica o el notebook debidamente comentados.

### Notas:
- Recuerda que debes establecer una sesión con Spark. Por defecto prueba en local. En la práctica puedes usar la conexión a Hadoop/Yarn como ejercicio opcional empleando el fichero de 600.000 líneas.


## Ejercicio 1 (p1_ecommerce.py)

Extracción, filtrado y almacenamiento de información usando el fichero `2019-Oct_10k.txt`.

Crear un script que haga lo siguiente:
 1. A partir del fichero `2019-Oct_10k.txt` obtener el número total de interacciones (de cualquier tipo de evento) que ha recibido cada producto. Debes obtener un DataFrame de la siguiente forma y guardarlo en el directorio `dfInteracciones.parquet`

| product_id | num_interacciones |
|------------|-----------------:|
| 2900536    | 6               |
| 1004739    | 42              |
| 23800009   | 3               |
| 1005158    | 6               |
| 12300850   | 1               |

 2. A partir del mismo fichero, crear un DataFrame que contenga el identificador del producto, la marca (brand) y el precio (price), descartando el resto de campos. Para evitar redundancias, debes eliminar las filas duplicadas basadas en el identificador del producto. Por último, renombra las columnas resultantes a ID_Producto, Marca y Precio. Ese DataFrame debe guardarse en el directorio `dfInfoProductos.parquet`
 y tener la siguiente forma:
 
|ID_Producto|  Marca|Precio|  
|-----------|-------|------|  
|    1002099|samsung|370.41|  
|    1002524|  apple|515.67|  
|    1002527|  apple|771.96|  
|    1002528|  apple|643.23|  
|    1002532|  apple|642.69|  


## Ejercicio 2 (p2_ecommerce.py)
Código Python que, a partir de los datos en Parquet generados en la práctica anterior (`dfInteracciones.parquet` y `dfInfoProductos.parquet`), obtenga para cada marca el total de productos distintos, el total de interacciones obtenidas por todos sus productos, la media de interacciones y el máximo número de interacciones conseguido por un solo producto.  
- Realizar el cruce de ambos ficheros quedándote solo con aquellos casos en los que existan valores en ambos DataFrames (inner join).  
- El DataFrame generado debe estar agrupado por la marca (Marca) y ordenado por el máximo número de interacciones (de forma descendente) y, en caso de empate, por el nombre de la marca (de forma ascendente).  
La salida del DataFrame antes de guardar en fichero debe tener este estilo:  

| Marca     | NumProductos | TotalInteracciones | MediaInteracciones | MaxInteracciones |
|-----------|-------------:|-----------------:|-----------------:|----------------:|
| samsung   | 180          | 1231             | 6.839            | 128             |
| apple     | 131          | 1039             | 7.931            | 112             |
| xiaomi    | 142          | 698              | 4.915            | 63              |
| huawei    | 50           | 306              | 6.120            | 33              |
| cordiant  | 22           | 89               | 4.045            | 31              |
| elari     | 5            | 29               | 5.800            | 18              |
| force     | 9            | 42               | 4.667            | 18              |
| acer      | 38           | 95               | 2.500            | 17              |
| elenberg  | 34           | 102              | 3.000            | 17              |
| luminarc  | 13           | 48               | 3.692            | 17              |

## Ejercicio 3 (p3_ecommerce.py)
Uso de funciones de ventana (Window Functions) para obtener el ranking de productos por marca.

A partir de los datos en Parquet generados en el Ejercicio 1 (`dfInteracciones.parquet` y `dfInfoProductos.parquet`), crear un script que realice lo siguiente:
1. Realizar un **join** de ambos DataFrames basándose en el identificador del producto.
2. Filtrar el resultado para incluir únicamente las marcas: **apple**, **samsung** y **xiaomi**.
3. Utilizar **Window Functions** para asignar un rango (ranking) a cada producto dentro de su marca, basándose en el número de interacciones de forma descendente. Se debe utilizar `dense_rank`.
4. El DataFrame final debe mostrar el nombre de la marca, el ID del producto, el número de interacciones y su posición en el ranking.
5. Los resultados deben estar ordenados por Marca (ascendente) y por el número de interacciones (descendente).
6. Guardar el resultado en un único fichero CSV con cabecera en el directorio `resultados_ej3_csv`.

La salida esperada debe tener la siguiente forma:

| Marca   | ID_Producto | num_interacciones | Rango |
|---------|-------------|------------------:|------:|
| apple   | 1004237     | 107               | 1     |
| apple   | 1005115     | 48                | 2     |
| samsung | 1004856     | 128               | 1     |
| samsung | 1004767     | 73                | 2     |
| xiaomi  | 1004233     | 63                | 1     |
| xiaomi  | 1002100     | 25                | 2     |

## Ejercicio 4 (p4_ecommerce.py)
Para este ejercicio vamos a analizar el volumen de tráfico por horas utilizando el fichero `2019-Oct_600k.txt`, pero queremos centrar nuestro estudio en comparar el rendimiento de las tres marcas tecnológicas más populares del dataset.
- Crear un script que obtenga un DataFrame que nos muestre el número total de interacciones asociadas únicamente a las marcas apple, samsung y xiaomi, desglosado por cada hora del día (extrayendo la hora de la columna event_time).
- El DataFrame generado tiene que tener la siguiente forma:

| Marca   | Hora | NInteracciones |
|---------|----:|---------------:|
| apple   |   2 |            106 |
| apple   |   3 |             10 |
| apple   |   4 |           2210 |
| apple   |   5 |           4531 |
| apple   |   6 |           5592 |
| apple   |   7 |           6681 |
| apple   |   8 |           6903 |
| apple   |   9 |           7814 |
| apple   |  10 |           7873 |
| apple   |  11 |           7702 |
| apple   |  12 |           7412 |
| apple   |  13 |           6206 |
| samsung |   2 |            146 |
| samsung |   3 |             21 |
| samsung |   4 |           2685 |
| samsung |   5 |           5884 |
| samsung |   6 |           6492 |
| samsung |   7 |           7687 |
| samsung |   8 |           7962 |
| samsung |   9 |           9050 |


### Requisitos
* Debes cargar el fichero 2019-Oct_600k.txt.
* Debes filtrar los registros para quedarte exclusivamente con aquellos donde la marca (brand) sea apple, samsung o xiaomi. (Pista: puedes usar el método .isin()).
* Para extraer la hora, utiliza la función hour() del módulo pyspark.sql.functions.
* El DataFrame debe estar agrupado por marca y hora.
* El resultado debe estar ordenado por la marca de forma ascendente y por la hora de forma ascendente.
* La salida debe guardarse en un único fichero CSV sin comprimir y con cabecera.